# StackGAN-v2 vs SDXL Turbo

Computer Vision course project. Side-by-side text-to-image comparison between a 2017 class-conditional GAN (StackGAN-v2, CUB birds) and a 2023 distilled diffusion model (SDXL Turbo, any text).

This notebook contains every piece of the project in one place: model code, inference wrapper, SDXL pipeline, weight downloader, and the Gradio app.

**Before running on Colab:** Runtime &rarr; Change runtime type &rarr; **T4 GPU**.

## 1. Install dependencies

In [ ]:
!pip install -q torch torchvision diffusers transformers accelerate gradio huggingface_hub Pillow numpy gdown kaggle

## 2. Imports

In [ ]:
import os
import json
import pickle
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 3. StackGAN-v2 generator architecture

Three-stage progressive generator. Takes a 1024-d text embedding and a 100-d noise vector, returns images at 64, 128, and 256 px. We only use the 256 px output.

- `CA_NET` projects the text embedding into a 128-d conditioning code with a small variational trick.
- Each `STAGE_G` block does joint conv &rarr; residual blocks &rarr; 2x upsample.
- `GLU` (Gated Linear Unit) replaces ReLU throughout.

In [ ]:
Z_DIM = 100
EMBEDDING_DIM = 128
GF_DIM = 64
R_NUM = 2
TEXT_DIM = 1024


def conv3x3(in_ch, out_ch):
    return nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False)


class GLU(nn.Module):
    # Split channels in half, gate one half with sigmoid of the other.
    def forward(self, x):
        nc = x.size(1) // 2
        return x[:, :nc] * torch.sigmoid(x[:, nc:])


def upBlock(in_ch, out_ch):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="nearest"),
        conv3x3(in_ch, out_ch * 2),
        nn.BatchNorm2d(out_ch * 2),
        GLU(),
    )


def Block3x3_relu(in_ch, out_ch):
    return nn.Sequential(
        conv3x3(in_ch, out_ch * 2),
        nn.BatchNorm2d(out_ch * 2),
        GLU(),
    )


class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            conv3x3(ch, ch * 2), nn.BatchNorm2d(ch * 2), GLU(),
            conv3x3(ch, ch), nn.BatchNorm2d(ch),
        )

    def forward(self, x):
        return x + self.block(x)

In [ ]:
class CA_NET(nn.Module):
    # Text embedding -> (mu, logvar) -> sampled conditioning code.
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(TEXT_DIM, EMBEDDING_DIM * 4, bias=True)
        self.relu = GLU()

    def encode(self, text_emb):
        x = self.relu(self.fc(text_emb))
        return x[:, :EMBEDDING_DIM], x[:, EMBEDDING_DIM:]

    def reparametrize(self, mu, logvar):
        std = logvar.mul(0.5).exp_()
        return torch.randn_like(std).mul(std).add_(mu)

    def forward(self, text_emb):
        mu, logvar = self.encode(text_emb)
        return self.reparametrize(mu, logvar), mu, logvar


class INIT_STAGE_G(nn.Module):
    # Stage 1: noise + conditioning -> 64x64 feature map.
    def __init__(self, ngf):
        super().__init__()
        self.gf_dim = ngf
        self.fc = nn.Sequential(
            nn.Linear(Z_DIM + EMBEDDING_DIM, ngf * 4 * 4 * 2, bias=False),
            nn.BatchNorm1d(ngf * 4 * 4 * 2),
            GLU(),
        )
        self.upsample1 = upBlock(ngf, ngf // 2)
        self.upsample2 = upBlock(ngf // 2, ngf // 4)
        self.upsample3 = upBlock(ngf // 4, ngf // 8)
        self.upsample4 = upBlock(ngf // 8, ngf // 16)

    def forward(self, z_code, c_code):
        x = torch.cat((c_code, z_code), 1)
        x = self.fc(x).view(-1, self.gf_dim, 4, 4)
        return self.upsample4(self.upsample3(self.upsample2(self.upsample1(x))))


class NEXT_STAGE_G(nn.Module):
    # Stage 2 and 3: refine + upsample, re-inject the text conditioning.
    def __init__(self, ngf, num_residual=R_NUM):
        super().__init__()
        self.jointConv = Block3x3_relu(ngf + EMBEDDING_DIM, ngf)
        self.residual = nn.Sequential(*[ResBlock(ngf) for _ in range(num_residual)])
        self.upsample = upBlock(ngf, ngf // 2)

    def forward(self, h_code, c_code):
        s = h_code.size(2)
        c = c_code.view(-1, EMBEDDING_DIM, 1, 1).repeat(1, 1, s, s)
        x = self.jointConv(torch.cat((c, h_code), 1))
        return self.upsample(self.residual(x))


class GET_IMAGE_G(nn.Module):
    def __init__(self, ngf):
        super().__init__()
        self.img = nn.Sequential(conv3x3(ngf, 3), nn.Tanh())

    def forward(self, h_code):
        return self.img(h_code)


class G_NET(nn.Module):
    # Full 3-stage generator. Returns [64, 128, 256] images; we keep the last.
    def __init__(self):
        super().__init__()
        self.ca_net = CA_NET()
        self.h_net1 = INIT_STAGE_G(GF_DIM * 16)
        self.img_net1 = GET_IMAGE_G(GF_DIM)
        self.h_net2 = NEXT_STAGE_G(GF_DIM)
        self.img_net2 = GET_IMAGE_G(GF_DIM // 2)
        self.h_net3 = NEXT_STAGE_G(GF_DIM // 2)
        self.img_net3 = GET_IMAGE_G(GF_DIM // 4)

    def forward(self, z_code, text_emb):
        c_code, mu, logvar = self.ca_net(text_emb)
        h1 = self.h_net1(z_code, c_code)
        h2 = self.h_net2(h1, c_code)
        h3 = self.h_net3(h2, c_code)
        return [self.img_net1(h1), self.img_net2(h2), self.img_net3(h3)], mu, logvar

## 4. Download weights and CUB embeddings

- StackGAN-v2 generator weights live on Google Drive (~80 MB). The download sometimes arrives as a zip rather than a raw `.pth`, so we detect that and unzip on the fly.
- CUB text embeddings live on a Kaggle mirror (the original Drive link is dead). Kaggle needs an API token; if it isn't available we skip the download and use a synthetic-embedding fallback during inference.

In [ ]:
import gdown

GDRIVE_WEIGHTS_ID = "1s5Yf3nFiXx0lltMFOiJWB6s1LP24RcwH"
KAGGLE_DATASET = "somthirthabhowmk2001/text-to-image-cub-200-2011"

HERE = Path.cwd()
WEIGHTS_PATH = HERE / "weights" / "netG_210000.pth"
EMBEDDINGS_PICKLE = HERE / "embeddings" / "char-CNN-RNN-embeddings.pickle"
KAGGLE_CACHE = HERE / ".cache" / "kaggle"


def download_stackgan_weights():
    if WEIGHTS_PATH.exists():
        print(f"  [skip] {WEIGHTS_PATH}")
        return
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    url = f"https://drive.google.com/uc?id={GDRIVE_WEIGHTS_ID}"
    gdown.download(url, str(WEIGHTS_PATH), quiet=False)

    # Drive sometimes returns a zip with netG_210000.pth inside.
    with open(WEIGHTS_PATH, "rb") as f:
        is_zip = f.read(4) == b"PK\x03\x04"
    if is_zip:
        archive = WEIGHTS_PATH.with_suffix(".pth.zip")
        WEIGHTS_PATH.rename(archive)
        with zipfile.ZipFile(archive) as zf:
            inner = next(n for n in zf.namelist() if n.endswith("netG_210000.pth"))
            with zf.open(inner) as src, open(WEIGHTS_PATH, "wb") as dst:
                dst.write(src.read())
        archive.unlink()
        print(f"  [unzip] -> {WEIGHTS_PATH}")


def _have_kaggle_token():
    # Check the standard locations the kaggle library reads from.
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    for p in (Path.home() / ".kaggle" / "kaggle.json",
              Path("/root/.kaggle/kaggle.json"),
              Path("/kaggle/input/kaggle.json")):
        if p.exists():
            return True
    return False


def download_embeddings():
    if EMBEDDINGS_PICKLE.exists():
        print(f"  [skip] {EMBEDDINGS_PICKLE}")
        return
    # The kaggle library calls exit(1) on missing creds, which we can't catch
    # reliably across versions. So we check for a token upfront and bail early.
    if not _have_kaggle_token():
        print("  No kaggle.json found. Skipping embeddings download.")
        print("  The app will use the synthetic-embedding fallback (still produces birds).")
        return
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        KAGGLE_CACHE.mkdir(parents=True, exist_ok=True)
        api.dataset_download_files(KAGGLE_DATASET, path=str(KAGGLE_CACHE),
                                   quiet=False, unzip=True)
    except BaseException as e:
        print(f"  Kaggle download failed: {type(e).__name__}: {e}")
        print("  Falling back to synthetic embeddings.")
        return
    found = (
        next(KAGGLE_CACHE.rglob("test/char-CNN-RNN-embeddings.pickle"), None)
        or next(KAGGLE_CACHE.rglob("char-CNN-RNN-embeddings.pickle"), None)
    )
    if found is None:
        print("  embeddings pickle not found in dataset")
        return
    EMBEDDINGS_PICKLE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(found, EMBEDDINGS_PICKLE)
    print(f"  [copy] -> {EMBEDDINGS_PICKLE}")


download_stackgan_weights()
download_embeddings()

## 5. StackGAN inference wrapper

Loads the generator, optionally loads the CUB embeddings pickle, and exposes a `generate` method that returns a PIL image.

If the embeddings file is missing, the wrapper falls back to a deterministic random embedding. StackGAN's bird prior is strong enough that the output still looks like a real bird.

In [ ]:
class StackGANInference:
    def __init__(self, weights_path, embeddings_path=None, captions=None, device="cpu"):
        self.device = torch.device(device)
        self._load_embeddings(embeddings_path)
        self._load_model(weights_path)
        self.captions = captions or []

    def _load_embeddings(self, path):
        self.embeddings = None
        if path and Path(path).exists():
            with open(path, "rb") as f:
                emb = np.asarray(pickle.load(f, encoding="latin1"), dtype=np.float32)
            self.embeddings = emb

    def _load_model(self, path):
        netG = G_NET()
        sd = torch.load(path, map_location="cpu", weights_only=False)
        if isinstance(sd, dict) and "state_dict" in sd:
            sd = sd["state_dict"]
        # Strip the DataParallel 'module.' prefix from training checkpoints.
        cleaned = {(k.replace("module.", "", 1) if k.startswith("module.") else k): v
                   for k, v in sd.items()}
        netG.load_state_dict(cleaned, strict=False)
        self.netG = netG.eval().to(self.device)

    def caption_labels(self):
        return [c["label"] for c in self.captions]

    def lookup_caption(self, label):
        for c in self.captions:
            if c["label"] == label:
                return c
        raise KeyError(label)

    def _embedding_for(self, image_idx, caption_idx):
        if self.embeddings is not None:
            emb = self.embeddings[image_idx, caption_idx, :]
            return torch.from_numpy(emb).float().unsqueeze(0).to(self.device)
        # Deterministic synthetic fallback.
        g = torch.Generator()
        g.manual_seed(int(image_idx) * 100003 + int(caption_idx) * 1009 + 17)
        return torch.randn(1, 1024, generator=g).to(self.device) * 0.5

    @torch.no_grad()
    def generate(self, image_idx, caption_idx=0, seed=None):
        emb = self._embedding_for(image_idx, caption_idx)
        g = torch.Generator()
        if seed is not None:
            g.manual_seed(int(seed))
        z = torch.randn(1, 100, generator=g).to(self.device)
        fake_imgs, _, _ = self.netG(z, emb)
        img = fake_imgs[-1][0]  # take the 256x256 output
        arr = ((img + 1) / 2 * 255).clamp(0, 255).byte().permute(1, 2, 0).cpu().numpy()
        return Image.fromarray(arr)

    def generate_by_label(self, label, seed=None):
        c = self.lookup_caption(label)
        return self.generate(c["image_idx"], c.get("caption_idx", 0), seed=seed)

## 6. SDXL Turbo pipeline

Thin wrapper over `diffusers.AutoPipelineForText2Image`. Defaults to `stabilityai/sdxl-turbo` (ungated, 4-step generation, ~2 s on T4). VAE tiling is enabled to keep peak VRAM under the T4's 16 GB.

In [ ]:
DEFAULT_SD_MODEL_ID = "stabilityai/sdxl-turbo"


class SDPipeline:
    def __init__(self, model_id=DEFAULT_SD_MODEL_ID, device="cuda"):
        self.model_id = model_id
        self.device = torch.device(device)
        self._pipe = None

    def _ensure_loaded(self):
        if self._pipe is not None:
            return
        from diffusers import AutoPipelineForText2Image
        dtype = torch.float16 if self.device.type == "cuda" else torch.float32
        token = os.environ.get("HF_TOKEN")
        pipe = AutoPipelineForText2Image.from_pretrained(
            self.model_id,
            torch_dtype=dtype,
            safety_checker=None,
            requires_safety_checker=False,
            token=token,
            variant="fp16" if dtype == torch.float16 else None,
        )
        pipe = pipe.to(self.device)
        try:
            pipe.enable_vae_tiling()  # keeps VRAM in check on T4
        except Exception:
            pass
        self._pipe = pipe

    def generate(self, prompt, seed=None, height=512, width=512):
        self._ensure_loaded()
        gen = (torch.Generator(device=self.device).manual_seed(int(seed))
               if seed is not None else None)
        is_turbo = "turbo" in self.model_id.lower()
        with torch.inference_mode():
            out = self._pipe(
                prompt=prompt,
                num_inference_steps=4 if is_turbo else 25,
                guidance_scale=0.0 if is_turbo else 7.5,
                generator=gen,
                height=height,
                width=width,
            )
        return out.images[0]

## 7. Quick test

Generate one image with each model so we can confirm both pipelines work before launching the UI.

In [ ]:
DROPDOWN_CAPTIONS = [
    {"label": "CUB Bird #1 (index 0)",    "image_idx": 0,    "caption_idx": 0},
    {"label": "CUB Bird #2 (index 1000)", "image_idx": 1000, "caption_idx": 0},
    {"label": "CUB Bird #3 (index 2700)", "image_idx": 2700, "caption_idx": 0},
]

stackgan = StackGANInference(
    weights_path=str(WEIGHTS_PATH),
    embeddings_path=str(EMBEDDINGS_PICKLE) if EMBEDDINGS_PICKLE.exists() else None,
    captions=DROPDOWN_CAPTIONS,
    device=DEVICE,
)

t0 = time.time()
bird = stackgan.generate_by_label("CUB Bird #1 (index 0)", seed=42)
print(f"StackGAN: {time.time() - t0:.2f}s")
bird

In [ ]:
sd_pipe = SDPipeline(device=DEVICE)

t0 = time.time()
img = sd_pipe.generate("a small yellow songbird singing on a green leaf, photo", seed=42)
print(f"SDXL Turbo: {time.time() - t0:.2f}s")
img

## 8. Gradio app

Side-by-side UI: pick a CUB caption from the dropdown on the left, type a free-text prompt for SDXL Turbo on the right, hit Generate. On Colab, set `share=True` to get a public URL.

Stop the cell to shut the app down.

In [ ]:
import gradio as gr

EXAMPLE_PROMPTS = [
    "a small yellow songbird singing on a green leaf, photo",
    "a great horned owl on a tree at night, cinematic photo",
    "a photograph of a cardinal perched on a snowy branch",
    "a corgi running through a sunflower field, dslr photo",
    "an astronaut riding a horse on Mars, digital art",
    "a peacock spreading its tail feathers, golden hour light",
]

caption_choices = stackgan.caption_labels()
default_caption = caption_choices[0] if caption_choices else None


def run_stackgan(label, seed):
    if not label:
        return None, "Pick a caption from the dropdown."
    t0 = time.time()
    img = stackgan.generate_by_label(label, seed=int(seed))
    return img, f"StackGAN-v2 - {time.time() - t0:.1f}s - 256x256"


def run_sd(prompt, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        return None, "Type a prompt."
    t0 = time.time()
    try:
        img = sd_pipe.generate(prompt, seed=int(seed))
    except Exception as e:
        return None, f"SDXL Turbo error: {e}"
    return img, f"SDXL Turbo - {time.time() - t0:.1f}s - 512x512"


def run_both(label, prompt, seed):
    sg_img, sg_info = run_stackgan(label, seed)
    sd_img, sd_info = run_sd(prompt, seed)
    return sg_img, sg_info, sd_img, sd_info


with gr.Blocks(title="StackGAN-v2 vs SDXL Turbo") as demo:
    gr.Markdown(
        "# StackGAN-v2 vs SDXL Turbo\n\n"
        "StackGAN-v2 (2017) only draws birds from CUB - pick from the dropdown. "
        "SDXL Turbo (2023) accepts any free text. Same seed, side by side."
    )
    with gr.Row():
        with gr.Column(scale=1):
            cap_dropdown = gr.Dropdown(
                choices=caption_choices, value=default_caption,
                label="StackGAN caption (CUB)",
            )
            free_text = gr.Textbox(
                label="SDXL Turbo prompt",
                value=EXAMPLE_PROMPTS[0], lines=2,
            )
            seed_in = gr.Number(label="Seed", value=42, precision=0)
            go = gr.Button("Generate", variant="primary")
            gr.Examples([[p] for p in EXAMPLE_PROMPTS], inputs=[free_text])
        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### StackGAN-v2 (2017, CUB only)")
                    sg_image = gr.Image(label="output", type="pil")
                    sg_info = gr.Markdown("Pick a caption and press Generate.")
                with gr.Column():
                    gr.Markdown("### SDXL Turbo (2023, any text)")
                    sd_image = gr.Image(label="output", type="pil")
                    sd_info = gr.Markdown("Type a prompt and press Generate.")
    go.click(run_both, inputs=[cap_dropdown, free_text, seed_in],
             outputs=[sg_image, sg_info, sd_image, sd_info])

demo.launch(share=True)

## Done

That's the whole project in one notebook: generator architecture, weight loading, both pipelines, and the comparison UI.

Repo: https://github.com/Stack-Gen-CV-Project/StackGAN